# Adjudication Impact Deep Dive

Notebook 11 (`11_affinity_diagnostisc_metrics_pipeline.ipynb`) showed the headline effect of LLM
adjudication: what fraction of judged pairs moved closer to or further from the expert label. This
notebook picks up from that last section and looks at the individual cases behind those numbers.

For every `(Abstract_Index, RA2025_ID)` pair sent to the judge (`Final_Method == "adjudicated_llm"`),
we compare:
- `prejudge_level` — the bucket implied by the raw cross-model mean score (`Affinity_Mean`)
- `pred_level` — the bucket implied by the judge's final score (`Final_Affinity`)
- `expert_level` — the ground-truth bucket from the expert calibration set

Two dataframes are built:
1. `improved_df` — pairs where the judge moved the label from wrong to correct
2. `worsened_df` — pairs where the judge moved the label from correct to wrong

Each row is enriched with the abstract text, the RA question, every individual model's raw score
and reasoning, and the judge's own `Final_Reason`, so specific cases can be read end-to-end.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_colwidth", 200)
plt.rcParams["figure.dpi"] = 120

## Load expert labels, adjudicated predictions, per-model votes, and abstract text

In [ ]:
repo_root = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()

expert_root = repo_root / "data" / "processed" / "affinity_calibration"
pred_root = repo_root / "data" / "results" / "affinity_benchmark"

expert_run_dir = "20260728_abstracts_evaluation_template_mark_om_v1"
# pred_run_dir = "20260726_134607_1"
pred_run_dir = "20260726_134607_5"

expert_label_col = "Evaluator_Affinity_Level"
pred_score_col = "Final_Affinity"

LABEL_ORDER = ["Low", "Moderate", "High"]
LABEL_TO_NUM = {"Low": 0, "Moderate": 1, "High": 2}


def normalize_level(value):
    if pd.isna(value):
        return np.nan
    if isinstance(value, str):
        text = value.strip().lower()
        return text.capitalize() if text in {"low", "moderate", "high"} else np.nan
    score = float(value)
    if 0 <= score < 40:
        return "Low"
    if 40 <= score <= 80:
        return "Moderate"
    if 80 <= score < 120:
        return "High"
    return np.nan

In [ ]:
expert_df = pd.read_csv(expert_root / expert_run_dir / "calibration_abstract_question_affinities.csv")
pred_df = pd.read_csv(pred_root / pred_run_dir / "adjudicated_ra_affinities.csv")
model_rows_df = pd.read_csv(pred_root / pred_run_dir / "adjudicated_ra_model_rows.csv")
abstract_lookup_df = pd.read_csv(pred_root / pred_run_dir / "abstract_lookup_cleaned.csv")

print("Expert rows:", len(expert_df))
print("Pred rows:", len(pred_df))
print("Model-vote rows:", len(model_rows_df))
print("Abstracts with cleaned text:", len(abstract_lookup_df))

## Build the pre- vs post-judge comparison (same logic as notebook 11)

In [ ]:
# expert_df and pred_df both carry 'Document Title' / 'RA_Question'; keep pred_df's copies only.
expert_cols = ["Abstract_Index", "RA2025_ID", expert_label_col]
merged_df = pd.merge(expert_df[expert_cols], pred_df, on=["Abstract_Index", "RA2025_ID"], how="inner")
merged_df["expert_level"] = merged_df[expert_label_col].apply(normalize_level)
merged_df["prejudge_level"] = merged_df["Affinity_Mean"].apply(normalize_level)
merged_df["pred_level"] = merged_df[pred_score_col].apply(normalize_level)

clean_df = merged_df.dropna(subset=["expert_level", "prejudge_level", "pred_level"]).copy()
for col in ["expert_level", "prejudge_level", "pred_level"]:
    clean_df[col.replace("_level", "_num")] = clean_df[col].map(LABEL_TO_NUM)

# Only pairs actually sent to the LLM judge: ensemble_mean pairs have prejudge == pred by
# construction and would just show up as "unchanged", adding no information here.
adjudicated_df = clean_df[clean_df["Final_Method"] == "adjudicated_llm"].copy()

adjudicated_df["prejudge_matches_expert"] = adjudicated_df["prejudge_num"] == adjudicated_df["expert_num"]
adjudicated_df["pred_matches_expert"] = adjudicated_df["pred_num"] == adjudicated_df["expert_num"]
adjudicated_df["judge_helped"] = (
    adjudicated_df["pred_matches_expert"].astype(int) - adjudicated_df["prejudge_matches_expert"].astype(int)
)
adjudicated_df["shift_size"] = (adjudicated_df["pred_num"] - adjudicated_df["prejudge_num"]).abs()

print("Judged pairs:", len(adjudicated_df))
print(adjudicated_df["judge_helped"].map({1: "improved", 0: "unchanged", -1: "worsened"}).value_counts())

## Enrich each judged pair with abstract text, RA question, and per-model votes

`adjudicated_ra_affinities.csv` already carries `Document Title` and `RA_Question`. We add:
- `Abstract_Cleaned` — the cleaned abstract text, from `abstract_lookup_cleaned.csv`
- one `score__<model_slug>` column per benchmarked model, from `adjudicated_ra_model_rows.csv`,
  showing what each model scored the pair before the judge saw it

In [ ]:
model_votes_df = model_rows_df.pivot_table(
    index=["Abstract_Index", "RA2025_ID"],
    columns="Model_Slug",
    values="LLM_Affinity",
    aggfunc="first",
).add_prefix("score__")

enriched_df = (
    adjudicated_df
    .merge(abstract_lookup_df[["Abstract_Index", "Abstract_Cleaned"]], on="Abstract_Index", how="left")
    .merge(model_votes_df, on=["Abstract_Index", "RA2025_ID"], how="left")
)

enriched_df.shape

## 1) Cases where adjudication improved the classification

Pairs where the pre-judge bucket disagreed with the expert but the judge's final bucket matched it.

In [ ]:
summary_cols = [
    "Abstract_Index", "RA2025_ID", "Document Title", "RA_Question",
    "expert_level", "prejudge_level", "pred_level",
    "Affinity_Mean", "Affinity_Std", "Affinity_Range", "agreement_mean", "shift_size",
]

improved_df = (
    enriched_df[enriched_df["judge_helped"] > 0]
    .sort_values("shift_size", ascending=False)
    .reset_index(drop=True)
)

print(f"{len(improved_df)} pairs improved by adjudication")
display(improved_df[summary_cols].head(20))

## 2) Cases where adjudication worsened the classification

Pairs where the pre-judge bucket already matched the expert but the judge's final bucket moved away
from it.

In [ ]:
worsened_df = (
    enriched_df[enriched_df["judge_helped"] < 0]
    .sort_values("shift_size", ascending=False)
    .reset_index(drop=True)
)

print(f"{len(worsened_df)} pairs worsened by adjudication")
display(worsened_df[summary_cols].head(20))

## Full case reports

For a qualitative read, `print_case_report` lays out one pair end-to-end: the RA question, the
abstract, every model's raw score and reasoning, and the judge's `Final_Reason` for overriding (or
not overriding) the panel.

In [ ]:
def print_case_report(row, model_rows_df):
    print(f"Abstract: {row['Document Title']}  ({row['Abstract_Index']})")
    print(f"RA Question [{row['RA2025_ID']}]: {row['RA_Question']}")
    print(f"Abstract text: {row['Abstract_Cleaned']}")
    print()
    print(f"Expert label:            {row['expert_level']}")
    print(f"Pre-judge label (mean):  {row['prejudge_level']}  (Affinity_Mean={row['Affinity_Mean']:.1f})")
    print(f"Post-judge label:        {row['pred_level']}  (Final_Affinity={row['Final_Affinity']:.1f})")
    print()
    print("Per-model votes:")
    votes = model_rows_df[
        (model_rows_df["Abstract_Index"] == row["Abstract_Index"])
        & (model_rows_df["RA2025_ID"] == row["RA2025_ID"])
    ][["Model_Slug", "LLM_Affinity", "affinity_level", "LLM_Affinity_Reason"]]
    display(votes)
    print()
    print(f"Judge's Final_Reason: {row['Final_Reason']}")
    print("=" * 120)

### Worsened cases (largest shifts first)

In [ ]:
for _, row in worsened_df.head(3).iterrows():
    print_case_report(row, model_rows_df)

### Improved cases (largest shifts first)

In [ ]:
for _, row in improved_df.head(3).iterrows():
    print_case_report(row, model_rows_df)

## Does adjudication behave differently when the panel already agreed?

If the judge tends to worsen cases where the raw panel was already tightly clustered
(low `Affinity_Std` / `Affinity_Range`), that would suggest it is overriding a fairly clear
signal rather than resolving genuine disagreement.

In [ ]:
outcome_col = enriched_df["judge_helped"].map({1: "improved", 0: "unchanged", -1: "worsened"})
outcome_col.name = "outcome"

panel_agreement_by_outcome = (
    enriched_df.groupby(outcome_col)[["Affinity_Std", "Affinity_Range", "agreement_mean"]]
    .mean()
    .reindex(["improved", "unchanged", "worsened"])
)

display(panel_agreement_by_outcome)

## Raw model-score distributions by outcome group

Every pair contributes up to 5 individual model scores (`LLM_Affinity`, 0-120). Here they are
pooled and grouped into the six outcomes that matter for the adjudication question:

|            | Not adjudicated | Adjudicated, bucket unchanged | Adjudicated, bucket changed |
|---|---|---|---|
| **Correctly classified**   | matched expert without a judge | judge saw it, agreed with the panel, matched expert | **Improved** — judge fixed a wrong panel mean |
| **Incorrectly classified** | missed expert without a judge  | judge saw it, agreed with the panel, missed expert  | **Worsened** — judge broke a correct panel mean |

Each histogram bar is colored by **that individual vote's own bucket** (Low/Moderate/High), not by
the pair's outcome — coloring by outcome instead would silently repaint a model's own Moderate vote
as High whenever the pair it belongs to ended up High, hiding the exact disagreement we're trying to
see. For the two adjudicated columns, the judge's own `Final_Affinity` values are drawn as an amber
rug beneath the axis, with a dashed mean line, so you can see where the judge's answer actually
landed relative to the panel it was reading.

In [ ]:
clean_df["was_adjudicated"] = clean_df["Final_Method"] == "adjudicated_llm"
clean_df["pred_matches_expert"] = clean_df["pred_num"] == clean_df["expert_num"]
clean_df["prejudge_matches_expert"] = clean_df["prejudge_num"] == clean_df["expert_num"]
clean_df["judge_helped"] = np.where(
    clean_df["was_adjudicated"],
    clean_df["pred_matches_expert"].astype(int) - clean_df["prejudge_matches_expert"].astype(int),
    np.nan,
)


def outcome_group(row):
    if not row["was_adjudicated"]:
        return "Not adjudicated - Correct" if row["pred_matches_expert"] else "Not adjudicated - Incorrect"
    if row["judge_helped"] > 0:
        return "Adjudicated - Improved"
    if row["judge_helped"] < 0:
        return "Adjudicated - Worsened"
    return "Adjudicated - Unchanged (correct)" if row["pred_matches_expert"] else "Adjudicated - Unchanged (incorrect)"


clean_df["outcome_group"] = clean_df.apply(outcome_group, axis=1)

pair_info = clean_df[["Abstract_Index", "RA2025_ID", "outcome_group", "expert_level", "pred_level", "Final_Affinity", "was_adjudicated"]]
score_votes = model_rows_df.merge(pair_info, on=["Abstract_Index", "RA2025_ID"], how="inner")
score_votes["vote_level"] = score_votes["LLM_Affinity"].apply(normalize_level)
score_votes["vote_matches_expert"] = score_votes["vote_level"] == score_votes["expert_level"]

clean_df["outcome_group"].value_counts()

**A coloring choice matters here.** Each pair contributes 5 votes, and coloring every vote by the
*pair's* final bucket (the natural first instinct) silently overwrites what individual models
actually said: a model that voted `70` (its own bucket is Moderate) inside a pair that ended up
`High` would be painted High, implying an agreement that never happened. The bars below are colored
by each **vote's own bucket** instead, so disagreement inside a group stays visible.

In [ ]:
# Sequential blue ramp (light -> dark) encodes the ordinal Low/Moderate/High bucket;
# amber is reserved for the judge's own Final_Affinity, kept visually distinct from the ramp.
LEVEL_COLORS = {"Low": "#86b6ef", "Moderate": "#2a78d6", "High": "#104281"}
JUDGE_COLOR = "#eb6834"
BOUNDARY_COLOR = "#898781"
MEAN_LINE_COLOR = "#52514e"

BINS = np.arange(0, 125, 5)

GRID_LAYOUT = [
    ("Correctly classified", "Not adjudicated - Correct", "Adjudicated - Unchanged (correct)", "Adjudicated - Improved"),
    ("Incorrectly classified", "Not adjudicated - Incorrect", "Adjudicated - Unchanged (incorrect)", "Adjudicated - Worsened"),
]
COLUMN_TITLES = ["Not adjudicated", "Adjudicated - bucket unchanged", "Adjudicated - bucket changed"]


def plot_outcome_panel(ax, group_name):
    group_votes = score_votes[score_votes["outcome_group"] == group_name]
    n_pairs = pair_info[pair_info["outcome_group"] == group_name].shape[0]

    scores_by_level = [
        group_votes.loc[group_votes["vote_level"] == level, "LLM_Affinity"]
        for level in LABEL_ORDER
    ]
    ax.hist(
        scores_by_level,
        bins=BINS,
        stacked=True,
        color=[LEVEL_COLORS[level] for level in LABEL_ORDER],
        edgecolor="white",
        linewidth=0.5,
    )

    for boundary in (40, 80):
        ax.axvline(boundary, color=BOUNDARY_COLOR, linestyle="--", linewidth=1)

    panel_mean = group_votes["LLM_Affinity"].mean()
    ax.axvline(panel_mean, color=MEAN_LINE_COLOR, linewidth=1.5)
    pct_votes_correct = 100 * group_votes["vote_matches_expert"].mean() if len(group_votes) else float("nan")
    stat_lines = [f"model mean={panel_mean:.0f}", f"votes matching expert={pct_votes_correct:.0f}%"]
    stat_colors = [MEAN_LINE_COLOR, "#0b0b0b"]

    is_adjudicated = group_votes["was_adjudicated"].any() if len(group_votes) else False
    if is_adjudicated:
        judge_values = pair_info.loc[pair_info["outcome_group"] == group_name, "Final_Affinity"]
        y0, y1 = ax.get_ylim()
        rug_y = -0.04 * y1
        ax.plot(judge_values, np.full(len(judge_values), rug_y), "|", color=JUDGE_COLOR, markersize=10, alpha=0.6, clip_on=False)
        judge_mean = judge_values.mean()
        ax.axvline(judge_mean, color=JUDGE_COLOR, linestyle=":", linewidth=1.5)
        stat_lines.append(f"judge mean={judge_mean:.0f}")
        stat_colors.append(JUDGE_COLOR)
        ax.set_ylim(y0, y1)

    # A fixed corner stat box (rather than labels anchored to the mean lines) avoids overlap
    # when the model mean and judge mean happen to sit close together on the x-axis.
    for i, (line, color) in enumerate(zip(stat_lines, stat_colors)):
        ax.text(
            0.97, 0.95 - 0.13 * i, line, transform=ax.transAxes,
            color=color, fontsize=8, ha="right", va="top",
            bbox=dict(facecolor="white", edgecolor="none", alpha=0.75, pad=1),
        )

    ax.set_title(f"{group_name}\n(N pairs={n_pairs}, N votes={len(group_votes)})", fontsize=9)
    ax.set_xlim(0, 120)
    ax.yaxis.set_major_locator(plt.MaxNLocator(integer=True))


fig, axes = plt.subplots(2, 3, figsize=(16, 8), sharex=True)

for row_idx, (row_label, *group_names) in enumerate(GRID_LAYOUT):
    for col_idx, group_name in enumerate(group_names):
        plot_outcome_panel(axes[row_idx, col_idx], group_name)
    axes[row_idx, 0].set_ylabel(f"{row_label}\nCount of model votes")

for col_idx, title in enumerate(COLUMN_TITLES):
    axes[0, col_idx].annotate(
        title, xy=(0.5, 1.28), xycoords="axes fraction", ha="center", fontsize=11, fontweight="bold",
    )

for ax in axes[-1, :]:
    ax.set_xlabel("Individual model affinity score")

level_handles = [plt.Rectangle((0, 0), 1, 1, color=LEVEL_COLORS[level]) for level in LABEL_ORDER]
judge_handles = [
    plt.Line2D([0], [0], color=MEAN_LINE_COLOR, linewidth=1.5, label="Panel (model) mean"),
    plt.Line2D([0], [0], color=JUDGE_COLOR, linestyle=":", linewidth=1.5, label="Judge mean / rug of Final_Affinity"),
]
fig.legend(
    level_handles + judge_handles,
    [f"Vote's own bucket: {level}" for level in LABEL_ORDER] + [h.get_label() for h in judge_handles],
    loc="lower center", ncol=5, bbox_to_anchor=(0.5, -0.04), frameon=False,
)

fig.suptitle("Individual model votes pooled by adjudication outcome, colored by each vote's own bucket", y=1.04, fontsize=13)
plt.tight_layout()
plt.show()

**Reading the grid, now that color and position agree:**

- **Not adjudicated - Correct**: 100% of votes match the expert — genuine unanimous agreement,
  which is why no judge was ever triggered.
- **Not adjudicated - Incorrect**: only 7% of votes match the expert. The panel was *unanimously
  wrong*, not divided — and because the adjudication trigger fires on cross-model disagreement, a
  confident-but-wrong consensus like this never reaches the judge at all. That's a blind spot the
  worsened/improved framing doesn't cover: 38 pairs are wrong and no judge ever got a chance to fix
  them.
- **Adjudicated, unchanged**: 78% of votes already matched the expert where the outcome stayed
  correct, vs. 23% where it stayed incorrect — the judge is inheriting the panel's accuracy here,
  not adding to it (the gray "model mean" and amber "judge mean" lines sit almost on top of each
  other in both panels).
- **Improved**: only 48% of individual votes matched the expert, i.e. the correct signal was
  genuinely a minority view that the raw mean buried — and the judge surfaced it.
- **Worsened**: 54% of votes matched the expert — a plurality of individual models were *right* —
  and the judge still picked the wrong bucket for every one of these 16 pairs. Visually, a
  moderate-colored (mid-blue) cluster sits just below a high-colored (dark-blue) cluster that
  straddles the 80-point line, and the amber rug of `Final_Affinity` is bimodal (a small group in
  the high-30s/low-40s, a much larger group at 82-92) — the judge is resolving genuine near-boundary
  splits, and resolving them upward more often than the underlying vote counts justify.

So the original claim holds, but more precisely than first stated: it isn't that adjudication is
broadly harmful, or that it ignores hedged reasoning in general — it's specifically that when votes
split close to the 80-point boundary, the judge tips toward High more often than the vote counts
support. The *bigger* accuracy gap in this run, though, is the 38 unanimously-wrong, never-adjudicated
pairs, which no prompt change to the adjudicator will touch since they never reach it.